In [0]:
df_logs = spark.read.csv(
    "/Volumes/workspace/default/dataset_streamings_databricks/logs_streaming.csv",
    header=True,
    inferSchema=True
)

display(df_logs)

In [0]:
#Visualizar o nosso esquema

df_logs.printSchema()

In [0]:
display(df_logs.select("watch_time_minutes"))

In [0]:
#Para corrigir, criamos uma nova célula e importamos col de pyspark.sql.functions

from pyspark.sql.functions import col

#Vamos construir a transformação com withColumn progressivamente para converter a coluna watch_time_minutes para inteiro

df_logs = df_logs.withColumn(
    "watch_time_minutes",
    col("watch_time_minutes").cast("double").cast("int")
)

In [0]:
#Verificando o esquema após a conversão
df_logs.printSchema()

In [0]:
display(df_logs.select("watch_time_minutes"))

### Primeiras transformações

In [0]:
df_logs.select(
    "user_id",
    "movie_id",
    "watch_time_minutes",
    "playback_status"
).display()

**Padronizando o status de reprodução:**

In [0]:
from pyspark.sql.functions import lower

df_logs = df_logs.withColumn(
    "playback_status",
    lower("playback_status")
)

In [0]:
df_logs.select(
    "playback_status"
).display()

**Criando a coluna de horas de exibição:**

In [0]:
from pyspark.sql.functions import col

#withColumn(...) para alterar ou adicionar a nova coluna
df_logs = df_logs.withColumn(
    "watch_time_hours",
    col("watch_time_minutes")/60
)

In [0]:
display(df_logs)

**Classificando a duração das obras:**

In [0]:
from pyspark.sql.functions import when

df_logs = df_logs.withColumn(
    "watch_category",
    when(
        col("watch_time_minutes") >= 120, "Longo"
    )
    .otherwise("Curto")
)

In [0]:
display(df_logs)

**Filtrando reproduções concluídas:**


In [0]:
df_logs.filter(
    (col('playback_status') == "completed")
).display()

**Adicionando critério de tempo visto maior que zero:**

In [0]:
df_logs.filter(
    (col('playback_status') == "completed") &
    (col('watch_time_minutes') > 0)
)
.display()

**Agregando por tipo de assinatura**

In [0]:
from pyspark.sql.functions import count, avg, sum

df_logs.groupBy("subscription_type")\
    .agg(
        count("*").alias("total_sessoes"),
        avg("watch_time_minutes").alias("media_minutos_assistidos"),
        sum("watch_time_minutes").alias("total_minutos_assistidos")
    )\
    .display()

**Ordenando grupos por total de sessões:**

In [0]:
df_logs.groupBy("subscription_type")\
    .agg(
        count("*").alias("total_sessoes"),
        avg("watch_time_minutes").alias("media_minutos_assistidos"),
    )\
    .orderBy("total_sessoes", ascending=False)\
    .display()

**Analisando por país e evidenciando inconsistências de padronização:**

In [0]:
df_logs.groupBy("country")\
    .agg(
        count("*").alias("total_sessoes"),
        avg("watch_time_minutes").alias("media_minutos_assistidos"),
    )\
    .orderBy("total_sessoes", ascending=False)\
    .display()